# News Recommendation — Popularity Debiased MIECL

## Required Drive folder structure
```
My Drive/
└── News Recc Code/
    ├── data_process.py
    ├── model.py
    ├── main.py
    ├── evaluate.py
    ├── dummy.txt              ← empty file, created by Cell 3
    └── dataset/
        ├── MINDsmall_train/
        │   ├── news.tsv
        │   ├── behaviors.tsv
        │   └── entity_embedding.vec
        ├── MINDsmall_dev/
        │   ├── news.tsv
        │   ├── behaviors.tsv
        │   └── entity_embedding.vec
        └── glove/
            └── glove.840B.300d.txt
```
Upload `data_process.py`, `model.py`, `main.py`, `evaluate.py` from your local
`News Recc Code/` folder into `My Drive/News Recc Code/` before running.


## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── CONFIGURE THIS ────────────────────────────────────────────────────────────
# Path to the folder on your Drive that contains the 4 .py files and dataset/
BASE = '/content/drive/MyDrive/News Recc Code'
# ──────────────────────────────────────────────────────────────────────────────

# Add the code folder to Python's path so imports work
import sys
sys.path.insert(0, BASE)
os.chdir(BASE)   # make relative paths in main.py resolve correctly

print('Working directory:', os.getcwd())
print('Files found:', os.listdir(BASE))

## Cell 2 — Install dependencies

In [ ]:
# NLTK punkt tokenizer (used in data_process.py)
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Verify PyTorch sees the GPU
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Cell 3 — Create dummy.txt (required by DataProcess)

In [ ]:
# main.py passes file6='dummy.txt' to DataProcess.
# It is never actually read, but the constructor stores it, so the file must exist.
dummy_path = os.path.join(BASE, 'dummy.txt')
if not os.path.exists(dummy_path):
    open(dummy_path, 'w').close()
    print('Created dummy.txt')
else:
    print('dummy.txt already exists')

## Cell 4 — Verify dataset paths

In [ ]:
# These are the exact paths main.py uses (relative to BASE / cwd)
DATASET = os.path.join(BASE, 'dataset')

paths_to_check = [
    os.path.join(DATASET, 'MINDsmall_train', 'news.tsv'),
    os.path.join(DATASET, 'MINDsmall_train', 'behaviors.tsv'),
    os.path.join(DATASET, 'MINDsmall_train', 'entity_embedding.vec'),
    os.path.join(DATASET, 'MINDsmall_dev',   'news.tsv'),
    os.path.join(DATASET, 'MINDsmall_dev',   'behaviors.tsv'),
    os.path.join(DATASET, 'MINDsmall_dev',   'entity_embedding.vec'),
    os.path.join(DATASET, 'glove', 'glove.840B.300d.txt'),
]

all_ok = True
for p in paths_to_check:
    exists = os.path.exists(p)
    status = '\u2713' if exists else '\u2717 MISSING'
    print(f'{status}  {p}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nAll dataset files found — ready to run.')
else:
    print('\nFix the missing files above before continuing.')

## Cell 5 — Patch file paths for Drive

Two things need patching:
1. `data_process.py` has two hardcoded `open('/content/drive/MyDrive/...')` lines for entity embeddings.
2. `main.py` has relative dataset paths and a Windows `preserve_dir` default.

This cell rewrites them to use your `BASE` path. Safe to run multiple times.

In [ ]:
# ── Patch data_process.py: entity embedding paths ────────────────────────────
dp_path = os.path.join(BASE, 'data_process.py')
with open(dp_path, 'r', encoding='utf-8') as f:
    src = f.read()

DATASET = os.path.join(BASE, 'dataset')

old_train_ent = "open('/content/drive/MyDrive/News Recc Code/dataset/MINDsmall_train/entity_embedding.vec'"
new_train_ent = f"open(r'{DATASET}/MINDsmall_train/entity_embedding.vec'"
old_dev_ent   = "open('/content/drive/MyDrive/News Recc Code/dataset/MINDsmall_dev/entity_embedding.vec'"
new_dev_ent   = f"open(r'{DATASET}/MINDsmall_dev/entity_embedding.vec'"

src = src.replace(old_train_ent, new_train_ent).replace(old_dev_ent, new_dev_ent)
if 'import os' not in src:
    src = 'import os\n' + src

with open(dp_path, 'w', encoding='utf-8') as f:
    f.write(src)
print('data_process.py  — entity paths patched')

# ── Patch main.py: dataset paths and preserve_dir ────────────────────────────
PRESERVE = os.path.join(BASE, 'outputs')
os.makedirs(PRESERVE, exist_ok=True)

main_path = os.path.join(BASE, 'main.py')
with open(main_path, 'r', encoding='utf-8') as f:
    main_src = f.read()

replacements = [
    ("file1 = 'MINDsmall_train/news.tsv'",
     f"file1 = r'{DATASET}/MINDsmall_train/news.tsv'"),
    ("file2 = 'MINDsmall_dev/news.tsv'",
     f"file2 = r'{DATASET}/MINDsmall_dev/news.tsv'"),
    ("file3 = 'MINDsmall_train/behaviors.tsv'",
     f"file3 = r'{DATASET}/MINDsmall_train/behaviors.tsv'"),
    ("file4 = 'MINDsmall_dev/behaviors.tsv'",
     f"file4 = r'{DATASET}/MINDsmall_dev/behaviors.tsv'"),
    ("file5 = 'glove/glove.840B.300d.txt'",
     f"file5 = r'{DATASET}/glove/glove.840B.300d.txt'"),
    ("file6 = 'dummy.txt'",
     f"file6 = r'{BASE}/dummy.txt'"),
    ("default='C:/Users/anany/Desktop/Ananya/2023/Estonia Projects/News Recc/MIECL-master'",
     f"default=r'{PRESERVE}'"),
]

for old, new in replacements:
    if old in main_src:
        main_src = main_src.replace(old, new)
        print(f'main.py  — patched: {old[:55]}...')
    else:
        print(f'main.py  — already done: {old[:55]}...')

with open(main_path, 'w', encoding='utf-8') as f:
    f.write(main_src)

print(f'\nOutputs will be saved to: {PRESERVE}')

## Cell 6 — Uncomment the training loop

The training loop in `main.py` is wrapped in `'''...'''`. This cell removes those delimiters so the script actually trains.

In [ ]:
main_path = os.path.join(BASE, 'main.py')
with open(main_path, 'r', encoding='utf-8') as f:
    main_src = f.read()

TRAIN_OPEN  = "        '''\n        for n_ep"
TRAIN_CLOSE = "        del train_candidate, train_user, train_label\n    '''"

if TRAIN_OPEN in main_src:
    main_src = main_src.replace(
        TRAIN_OPEN, "        \n        for n_ep"
    ).replace(
        TRAIN_CLOSE, "        del train_candidate, train_user, train_label\n"
    )
    with open(main_path, 'w', encoding='utf-8') as f:
        f.write(main_src)
    print('Training loop uncommented — ready to train.')
else:
    print('Training loop already active.')

## Cell 7 — Run Training + Validation

| Arg | What it controls | Recommended |
|-----|-----------------|-------------|
| `--num_epoch` | Epochs per dataset pass | 1–3 |
| `--num_dataset` | Dataset loop repetitions | 1 (quick test), 6 (full) |
| `--batch_size` | Reduce if OOM on T4 | 30 |
| `--alpha` | Weight for existing user CL loss | 1.0 |
| `--contrastive_mode` | Must be `USER` to enable both CL losses | USER |
| `--infonce_mode` | User-level CL mode (unchanged from original) | prototype_self |

> The new popularity-debiased CL loss is always active when `contrastive_mode=USER`, weighted at **β = 0.5**.

In [ ]:
!python "{BASE}/main.py" \
    --num_epoch 1 \
    --num_dataset 1 \
    --batch_size 30 \
    --hid_dim 400 \
    --num_head 20 \
    --num_prototype 5 \
    --alpha 1.0 \
    --dropout_rate 0.1 \
    --multi_rep_mode concat \
    --infonce_mode prototype_self \
    --contrastive_mode USER \
    --gnn_mode nogat \
    --agg_mode soft

## Cell 8 — View Results

In [ ]:
import glob
PRESERVE = os.path.join(BASE, 'outputs')
score_files = sorted(glob.glob(os.path.join(PRESERVE, 'scores_*.txt')))

if not score_files:
    print('No score files found. Run Cell 7 first.')
else:
    for sf in score_files:
        print(f'\n=== {os.path.basename(sf)} ===')
        with open(sf) as f:
            print(f.read())

## Cell 9 — (Optional) Eval only — load a saved checkpoint

Skip training and just evaluate a previously saved model.

In [ ]:
PRESERVE = os.path.join(BASE, 'outputs')
checkpoints = sorted(glob.glob(os.path.join(PRESERVE, 'model_*.pkl')))
print('Saved checkpoints:')
for c in checkpoints:
    print(f'  {os.path.basename(c)}  ({os.path.getsize(c)/1e6:.1f} MB)')